# Self-Driving Car Maze Navigation

In this notebook you will implement four classic robot navigation algorithms and test them in a Pygame simulation.

Each algorithm is implemented as a single function with the signature:

```python
def algorithm_navigate(car_x, car_y, car_angle,
                       goal_x, goal_y,
                       obstacles, car_width, car_height, dt)
    -> (new_x, new_y, new_angle)
```

| Parameter | Description |
|-----------|-------------|
| `car_x`, `car_y` | Current position of the car's center (pixels) |
| `car_angle` | Current heading in **radians** (0 = facing right, π/2 = facing up in math coords — note Pygame's y-axis is **flipped**: increasing y goes **down**) |
| `goal_x`, `goal_y` | Position of the goal (pixels) |
| `obstacles` | List of `(x, y, width, height)` rectangles |
| `car_width`, `car_height` | Dimensions of the car (pixels) |
| `dt` | Time step in seconds |

Return the **new** position and heading after one time step.

---
**Coordinate system reminder:** Pygame's y-axis increases downward.  
So to move "up" on screen you **decrease** `y`, and `math.atan2(-(dy), dx)` gives the angle in screen space.

---
After you implement all four functions, run the **Export** cell at the bottom to write them to `algorithms.py`, then launch `simulation.py` to test your work.

In [ ]:
import math
import numpy as np

# ── Shared helper ─────────────────────────────────────────────────────────────
def _angle_diff(target, current):
    """Return the signed shortest angular difference (target - current) in [-π, π]."""
    diff = target - current
    while diff >  math.pi: diff -= 2 * math.pi
    while diff < -math.pi: diff += 2 * math.pi
    return diff


def reset_algorithm_state():
    """Clear per-algorithm state (called on maze reset)."""
    for fn in [bug2_navigate, pure_pursuit_navigate,
               stanley_navigate, potential_field_navigate]:
        if hasattr(fn, '_state'):
            fn._state = None

---
## 1 · Bug2 Algorithm

### How it works

Bug2 is one of the simplest complete navigation algorithms.  It relies on two behaviours:

1. **Go-to-goal** — drive in a straight line from start **S** toward the goal **G** along the *M-line* (the straight line connecting S and G).
2. **Boundary following** — when an obstacle is detected, switch to tracing the obstacle boundary until the M-line is re-encountered at a point **closer to the goal** than where boundary following began.  Then switch back to go-to-goal.

The algorithm is provably complete (it always reaches the goal if one exists) for simple connected obstacles.

### Key variables to track
* The **M-line** — the line segment from your starting position to the goal.
* The **hit-point** — where the car first hits an obstacle.
* The **best distance** seen so far on the M-line — used to decide when to leave boundary-following mode.

### Implementation hints

1. Store state (mode, hit-point, best distance, etc.) in `bug2_navigate._state` so it persists across calls.
2. **Go-to-goal mode:**  
   - Compute the angle to the goal with `math.atan2`.  
   - Rotate toward it (clamp the turn rate) then move forward.  
   - Switch to boundary-following when an obstacle is sensed ahead.
3. **Boundary-following mode:**  
   - Keep the obstacle to one side (e.g. left wall-following: turn right when there is space, turn left when blocked).  
   - At each step check whether you are back on the M-line **and** closer to the goal than the hit-point; if so, switch back to go-to-goal.
4. Always clamp the new position inside the screen and reject positions that would put the car inside an obstacle.

In [ ]:
def bug2_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                  obstacles, car_width, car_height, dt):
    """
    Bug2 algorithm.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED          = 80.0   # pixels / second
    TURN_RATE      = 2.5    # radians / second (max steering rate)
    SENSOR_DIST    = 40.0   # pixels ahead to check for obstacles
    OBSTACLE_MARGIN = 18.0  # extra clearance around obstacle rectangles

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(bug2_navigate, '_state', None) is None:
        bug2_navigate._state = {
            'mode': 'go_to_goal',      # 'go_to_goal' or 'follow_boundary'
            'hit_point': None,         # (x, y) where obstacle was first hit
            'best_dist_on_mline': None,# best distance to goal seen on M-line
            'boundary_dir': 1,         # +1 = left-hand, -1 = right-hand
            'start_x': car_x,
            'start_y': car_y,
        }
    state = bug2_navigate._state

    # ── Helper functions ──────────────────────────────────────────────────────
    def point_in_obstacle(x, y):
        """Return True if (x, y) is inside any obstacle (with margin)."""
        # TODO: implement
        pass

    def obstacle_ahead(x, y, angle, dist):
        """Return True if there is an obstacle within `dist` pixels ahead."""
        # TODO: implement — sample at least two points along the heading
        pass

    def dist_to_goal(x, y):
        return math.hypot(goal_x - x, goal_y - y)

    def on_mline(x, y):
        """
        Return True if (x, y) is close to the M-line AND between start and goal.
        Hint: compute the perpendicular distance from (x, y) to the line segment
        from (state['start_x'], state['start_y']) to (goal_x, goal_y).
        """
        # TODO: implement
        pass

    # ── Go-to-goal mode ───────────────────────────────────────────────────────
    if state['mode'] == 'go_to_goal':
        # TODO:
        # 1. Compute target angle toward goal
        # 2. Check if obstacle is ahead; if so, record hit-point and switch mode
        # 3. Otherwise steer toward goal and move forward
        # 4. Return (new_x, new_y, new_angle)
        pass

    # ── Boundary-following mode ───────────────────────────────────────────────
    if state['mode'] == 'follow_boundary':
        # TODO:
        # 1. Steer along the obstacle boundary (wall-following)
        # 2. After each step, check if back on M-line and closer to goal
        # 3. If so, switch back to go_to_goal
        # 4. Return (new_x, new_y, new_angle)
        pass

    # Fallback — stay in place
    return car_x, car_y, car_angle

---
## 2 · Pure Pursuit

### How it works

Pure Pursuit is a **path-tracking** algorithm originally developed for autonomous vehicle steering.  It assumes a reference path is already available and steers the vehicle toward a **lookahead point** — a point on the path that lies a fixed distance *L* ahead of the current position.

The key insight is geometric: given the lookahead point, the required steering curvature is:

$$\kappa = \frac{2 \cdot d_y}{L^2}$$

where $d_y$ is the lateral offset of the lookahead point in the vehicle's local frame and $L$ is the lookahead distance.

### Steps

1. **Build a waypoint path** from start to goal that avoids obstacles.  A simple greedy planner (try heading directly toward the goal; if blocked, try angles offset by ±0.4 rad, ±0.8 rad, …) works well enough for this simulation.
2. **Find the lookahead point** — walk forward along the waypoint list until the cumulative distance exceeds `LOOKAHEAD`.
3. **Steer toward the lookahead point** — compute the bearing to it, then rotate the car at most `TURN_RATE · dt` radians toward that bearing.
4. Move forward at constant speed.

### Implementation hints

* Cache the waypoint list in `pure_pursuit_navigate._state` — recompute only when the state is cleared (maze reset).
* Advance `wp_index` whenever the car comes within `WAYPOINT_REACH` pixels of the current waypoint.
* If the new position lands inside an obstacle, clear the cached waypoints so a new path is planned next frame.

In [ ]:
def pure_pursuit_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                          obstacles, car_width, car_height, dt):
    """
    Pure Pursuit controller.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 90.0
    LOOKAHEAD       = 60.0   # pixels — distance to the lookahead point
    TURN_RATE       = 3.0    # max radians / second
    OBSTACLE_MARGIN = 20.0
    WAYPOINT_REACH  = 30.0   # advance waypoint when within this many pixels

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(pure_pursuit_navigate, '_state', None) is None:
        pure_pursuit_navigate._state = {
            'waypoints': None,
            'wp_index':  0,
        }
    state = pure_pursuit_navigate._state

    # ── Helper: point / path clearance ───────────────────────────────────────
    def point_clear(x, y):
        """Return True if (x, y) is not inside any obstacle."""
        # TODO: implement
        pass

    def path_clear(x1, y1, x2, y2, steps=10):
        """Return True if the straight line from (x1,y1) to (x2,y2) is obstacle-free."""
        # TODO: sample `steps` points along the segment and call point_clear
        pass

    # ── Waypoint planner ──────────────────────────────────────────────────────
    def build_waypoints():
        """
        Build a list of (x, y) waypoints from the car's current position to the goal.
        Use a greedy approach: at each step, try heading directly toward the goal;
        if blocked, try headings offset by ±0.4, ±0.8, ±1.2, ±1.6, π radians.
        Step size: 50 pixels.  Max iterations: 200.
        """
        # TODO: implement
        pass

    # ── Build path if needed ──────────────────────────────────────────────────
    if state['waypoints'] is None:
        state['waypoints'] = build_waypoints()
        state['wp_index']  = 0

    waypoints = state['waypoints']

    # ── Advance waypoint index ────────────────────────────────────────────────
    # TODO: while current waypoint is within WAYPOINT_REACH, advance wp_index

    # ── Find lookahead point ──────────────────────────────────────────────────
    # TODO: walk forward from wp_index until cumulative path distance >= LOOKAHEAD
    lookahead_x, lookahead_y = waypoints[-1]  # fallback: end of path

    # ── Steer toward lookahead point ──────────────────────────────────────────
    # TODO:
    # 1. Compute bearing to lookahead point
    # 2. Compute angular difference and clamp to TURN_RATE * dt
    # 3. Update angle, compute new_x / new_y
    # 4. If new position is inside obstacle, clear waypoints and return old pos
    # 5. Return (new_x, new_y, new_angle)
    pass

    return car_x, car_y, car_angle

---
## 3 · Stanley Controller

### How it works

The Stanley controller was used by Stanford's autonomous vehicle *Stanley* to win the 2005 DARPA Grand Challenge.  It combines two error terms:

1. **Heading error** $\psi_e$ — the difference between the car's heading and the tangent direction of the reference path.
2. **Cross-track error** $e$ — the signed perpendicular distance from the car to the nearest path segment.

The steering command is:

$$\delta = \psi_e + \arctan\!\left(\frac{k \cdot e}{v}\right)$$

where $k$ is a gain and $v$ is the vehicle speed.  At high speed the cross-track correction is small (smooth); at low speed it can be large (aggressive correction).

### Steps

1. Build the same greedy waypoint path as in Pure Pursuit.
2. Find the **closest path segment** to the car (or use the current `wp_index`).
3. Compute the **heading error**: `heading_err = _angle_diff(segment_angle, car_angle)`.
4. Compute the **cross-track error**: signed perpendicular distance from the car to the segment (positive = left of path).
5. Combine: `delta = heading_err + atan2(K * cross_track, SPEED)`.
6. Clamp `delta` to `TURN_RATE * dt` and update angle and position.

### Implementation hints

* The **cross-track error** sign convention matters — make sure positive cross-track drives the car back onto the path from the left side.
* Reuse the same greedy `build_waypoints` helper from Pure Pursuit.
* The gain `K = 2.0` and `SPEED = 85.0` are good starting values.

In [ ]:
def stanley_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                     obstacles, car_width, car_height, dt):
    """
    Stanley controller.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 85.0
    K               = 2.0    # cross-track gain
    TURN_RATE       = 3.5
    OBSTACLE_MARGIN = 20.0
    WAYPOINT_REACH  = 25.0

    # ── Persistent state ──────────────────────────────────────────────────────
    if getattr(stanley_navigate, '_state', None) is None:
        stanley_navigate._state = {
            'waypoints': None,
            'wp_index':  0,
        }
    state = stanley_navigate._state

    # ── Helpers ───────────────────────────────────────────────────────────────
    def point_clear(x, y):
        # TODO: implement
        pass

    def path_clear(x1, y1, x2, y2, steps=10):
        # TODO: implement
        pass

    def build_waypoints():
        """
        Same greedy planner as Pure Pursuit — build a list of (x, y) waypoints
        from the car to the goal, stepping 50 px at a time.
        """
        # TODO: implement
        pass

    # ── Build path if needed ──────────────────────────────────────────────────
    if state['waypoints'] is None:
        state['waypoints'] = build_waypoints()
        state['wp_index']  = 0

    waypoints = state['waypoints']

    # ── Advance waypoint index ────────────────────────────────────────────────
    # TODO: advance wp_index while within WAYPOINT_REACH

    # ── Compute errors ────────────────────────────────────────────────────────
    # Current target waypoint
    wp_index = min(state['wp_index'], len(waypoints) - 1)
    tx, ty   = waypoints[wp_index]

    # Previous waypoint (for segment direction)
    if wp_index > 0:
        px, py = waypoints[wp_index - 1]
    else:
        px, py = car_x, car_y

    # TODO:
    # 1. Compute segment angle: math.atan2(-seg_dy, seg_dx)  (note sign of dy)
    # 2. Compute heading error: _angle_diff(seg_angle, car_angle)
    # 3. Compute cross-track error (signed perpendicular distance)
    #    Hint: use the 2-D cross product of the segment unit vector and the
    #          vector from the segment start to the car.
    # 4. delta = heading_err + atan2(K * cross_track, SPEED)
    # 5. Clamp delta to TURN_RATE * dt
    # 6. Update angle, compute new_x / new_y
    # 7. If new position is inside an obstacle, clear waypoints and return old pos
    # 8. Return (new_x, new_y, new_angle)
    pass

    return car_x, car_y, car_angle

---
## 4 · Potential Field Controller

### How it works

Potential Field navigation treats the robot's workspace as a scalar potential field:

* The **goal** creates an **attractive** potential that decreases with distance — like a gravity well pulling the robot in.
* Each **obstacle** creates a **repulsive** potential that increases as the robot gets close — like a magnetic repulsion.

The robot follows the **negative gradient** of the total potential, which gives a force vector pointing (approximately) toward the goal while pushing away from obstacles.

$$\mathbf{F}_{att} = k_{att} \cdot (\mathbf{q}_{goal} - \mathbf{q})$$

$$\mathbf{F}_{rep} = k_{rep} \left(\frac{1}{d} - \frac{1}{d_0}\right) \frac{1}{d^2} \hat{\mathbf{d}} \quad \text{if } d < d_0$$

where $d$ is the distance to the nearest obstacle surface, $d_0$ is the influence radius, and $\hat{\mathbf{d}}$ is the unit vector **away** from the obstacle.

The total force is $\mathbf{F} = \mathbf{F}_{att} + \sum \mathbf{F}_{rep}$. Normalise it to get a direction, then move at constant speed.

### Known limitation

Potential fields can get stuck in **local minima** (places where attractive and repulsive forces exactly cancel). If this happens, add a small random perturbation or implement a simple escape strategy.

### Implementation hints

* For each obstacle rectangle, the **closest point** on the rectangle to the car is `(clamp(car_x, ox, ox+ow), clamp(car_y, oy, oy+oh))`.
* Use `K_ATT = 1.0`, `K_REP = 8000.0`, `D0 = 60.0` as starting values.
* Normalise the total force vector before using it as a heading target.
* If the new position lands inside an obstacle, try a small angular perturbation.

In [ ]:
def potential_field_navigate(car_x, car_y, car_angle, goal_x, goal_y,
                             obstacles, car_width, car_height, dt):
    """
    Potential Field controller.

    Returns (new_x, new_y, new_angle).
    """
    # ── Tunable constants ─────────────────────────────────────────────────────
    SPEED           = 75.0
    TURN_RATE       = 3.0
    K_ATT           = 1.0      # attractive gain
    K_REP           = 8000.0   # repulsive gain
    D0              = 60.0     # repulsion influence radius (pixels)
    OBSTACLE_MARGIN = 15.0

    # ── Attractive force ──────────────────────────────────────────────────────
    # TODO: compute att_x, att_y = K_ATT * (goal - car)
    att_x, att_y = 0.0, 0.0

    # ── Repulsive forces ──────────────────────────────────────────────────────
    rep_x, rep_y = 0.0, 0.0
    for obs in obstacles:
        ox, oy, ow, oh = obs
        # TODO:
        # 1. Find the closest point on this rectangle to the car
        # 2. Compute distance d from car to that closest point
        # 3. If d < D0, compute repulsive force magnitude and direction
        #    magnitude = K_REP * (1/d - 1/D0) / d^2
        #    direction = unit vector from closest point to car
        # 4. Accumulate into rep_x, rep_y
        pass

    # ── Combine and normalise ─────────────────────────────────────────────────
    # TODO:
    # force = att + rep
    # Normalise force to unit vector
    # target_angle = math.atan2(-force_y, force_x)   (note minus sign for Pygame y)

    # ── Steer and move ────────────────────────────────────────────────────────
    # TODO:
    # 1. Compute angle_diff = _angle_diff(target_angle, car_angle)
    # 2. new_angle = car_angle + clamp(angle_diff, TURN_RATE * dt)
    # 3. new_x = car_x + cos(new_angle) * SPEED * dt
    # 4. new_y = car_y - sin(new_angle) * SPEED * dt   (minus: Pygame y down)
    # 5. If inside obstacle, try a small perturbation angle and recompute
    # 6. Return (new_x, new_y, new_angle)
    pass

    return car_x, car_y, car_angle

---
## Export to `algorithms.py`

Run the cell below once you have implemented all four functions.  It extracts the function source code and writes `algorithms.py` in the same directory, which `simulation.py` imports.

Then open a terminal and run:

```bash
python simulation.py
```

Use the dropdown in the sidebar to select an algorithm, and the **Reset Maze** button to randomise obstacles.

In [ ]:
import inspect
import textwrap

functions = [
    bug2_navigate,
    pure_pursuit_navigate,
    stanley_navigate,
    potential_field_navigate,
    _angle_diff,
    reset_algorithm_state,
]

header = """import math
import numpy as np


"""

output_path = 'algorithms.py'

with open(output_path, 'w') as f:
    f.write(header)
    for fn in functions:
        src = inspect.getsource(fn)
        f.write(src)
        f.write('\n\n')

print(f'Written to {output_path}')
print('Now run:  python simulation.py')